# Exercício 20 — Empresa Autónoma Integrada

**Entrega sem ecrã:** código executável abaixo + documentação em `docs/`.


## 0. Ambiente e caminhos

In [ ]:
from pathlib import Path
import os
from dotenv import load_dotenv

EX_ROOT = Path.cwd().resolve()
REPO_ROOT = EX_ROOT.parent.parent

load_dotenv(REPO_ROOT / ".env", override=False)
load_dotenv(EX_ROOT / ".env", override=True)

if not (os.environ.get("GOOGLE_API_KEY") or os.environ.get("GEMINI_API_KEY")):
    raise RuntimeError("Defina GOOGLE_API_KEY no .env na raiz do repositório.")

print("OK — Repo:", REPO_ROOT)
print("OK — Exercício:", EX_ROOT)


## Pipeline integrador (protótipo)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.output_parsers import StrOutputParser
import os

llm = ChatGoogleGenerativeAI(
    model=(os.environ.get("GEMINI_MODEL") or "gemini-2.0-flash").replace("models/", ""),
    temperature=0.25,
)

pedido = "Cliente quer saber prazo contratual e abrir reclamação."

cls = (ChatPromptTemplate.from_template("Classifica intent numa palavra: info | reclamacao | misto\n{pedido}") | llm | StrOutputParser()).invoke({"pedido": pedido})
ctx = "Contrato: prazo resposta 5 dias úteis para pedidos escritos."
rag = (ChatPromptTemplate.from_template("Contexto:{ctx}\nResponde objetivamente:\n{pedido}") | llm | StrOutputParser()).invoke({"ctx": ctx, "pedido": pedido})
rev = (ChatPromptTemplate.from_template("Revisa lacunas:\n{rag}") | llm | StrOutputParser()).invoke({"rag": rag})
print({"classificacao": cls, "rag": rag, "revisao": rev})


## Referências
- `docs/` desta pasta — explicações detalhadas.
- `app/` — espelho API opcional.
